create training csv from the schedule generation

In [ ]:
%cd /Users/aliceliusyncrowin/Projects/tata-schedule-fetch
import json, os
import importlib
with open("local.settings.json", "r") as f:
    os.environ.update(json.load(f).get("Values", {}))
from flatten_schedule import db, extract_jsonb
import pandas as pd
import datetime
import importlib


/Users/aliceliusyncrowin/Projects/tata-schedule-fetch


In [8]:
# CSV structure:
# schedule_date_ist, prediction_for_time_block, prediction_for_period_end_ist, time_retrieved, total_net_schd_amount, remc_station_sch_amount
# TODO: Check the time block to time block ending in -
def datetime_to_time_block(dt: datetime.datetime) -> int:
    """
    Convert a datetime object to a time block index (1-96) based on 15-minute intervals.
    """
    tb = dt.hour * 4 + dt.minute // 15 # floor division -> rounds down to nearest integer
    if tb == 0:
        tb = 96
    return tb

def timeblock_to_period_end(schedule_date_ist: datetime.date, time_block: int) -> datetime.datetime:
    """
    Convert a time block index (1-96) to a period end timestamp.
    """
    total_min = time_block * 15
    day_offset = total_min // 1440
    min_in_day = total_min % 1440
    hour = min_in_day // 60
    minute = min_in_day % 60
    period_end_date = schedule_date_ist + datetime.timedelta(days=day_offset)
    return datetime.datetime.combine(period_end_date, datetime.time(hour, minute))

In [9]:
# CSV structure:
# schedule_date_ist, prediction_for_time_block, prediction_for_period_end_ist, time_retrieved, total_net_schd_amount, remc_station_sch_amount

# For every 30 minute interval, 
# starting tb = 1 (00:00-00:15), retrieve block tb + 8 and tb + 9 (total_net_schd_amount, remc_station_sch_amount)

# record schedule_date_ist for the row, prediction_for_time_block (thetime block of t+8 and t+9), 
# prediction_for_period_end_ist (the time of the end of the period for t+8 and t+9), time_retrieved (the time of retrieval, which t+8 and t+9 is based on), total_net_schd_amount (for t+8 and t+9), remc_station_sch_amount (t+8 and t+9)




In [10]:
# get all jsonb rows that exist 
importlib.reload(db)
conn = db.get_connection()
try:
    all_jsonb_rows = db.get_schedule_structured_rows(conn)
finally:
    conn.close()

In [11]:
len(all_jsonb_rows)
df_all_jsonb_rows = pd.DataFrame(all_jsonb_rows)

In [12]:
df_all_jsonb_rows

,schedule_date_ist,time_retrieved,period_end_ist,time_block,OA_REMC_OA_GNA,EMERGENCY_AS,SHORTFALL_AS,total_net_schd_amount,remc_station_sch_amount,remc_created_on_IST,remc_revision_no,full_schd_revision_no,schedule_published_time_ist
0,2026-06-03,2026-06-25 23:05:58.450269+00:00,2026-06-03 00:15:00,1,0.00,0.0,0.0,0.00,0.00,2026-06-03 23:45:12,125,141,2026-06-03 23:48:00
1,2026-06-03,2026-06-25 23:05:58.450269+00:00,2026-06-03 00:30:00,2,0.00,0.0,0.0,0.00,0.00,2026-06-03 23:45:12,125,141,2026-06-03 23:48:00
2,2026-06-03,2026-06-25 23:05:58.450269+00:00,2026-06-03 00:45:00,3,0.00,0.0,0.0,0.00,0.00,2026-06-03 23:45:12,125,141,2026-06-03 23:48:00
3,2026-06-03,2026-06-25 23:05:58.450269+00:00,2026-06-03 01:00:00,4,0.00,0.0,0.0,0.00,0.00,2026-06-03 23:45:12,125,141,2026-06-03 23:48:00
4,2026-06-03,2026-06-25 23:05:58.450269+00:00,2026-06-03 01:15:00,5,0.00,0.0,0.0,0.00,0.00,2026-06-03 23:45:12,125,141,2026-06-03 23:48:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
76027,2026-06-24,2026-06-24 00:45:00.613754+00:00,2026-06-24 22:00:00,88,0.00,0.0,0.0,0.00,0.00,2026-06-24 06:00:13,56,72,2026-06-24 06:03:00
76028,2026-06-24,2026-06-24 00:45:00.613754+00:00,2026-06-24 14:15:00,57,-275.74,0.0,0.0,-275.74,275.74,2026-06-24 06:00:13,56,72,2026-06-24 06:03:00
76029,2026-06-24,2026-06-24 00:45:00.613754+00:00,2026-06-24 10:30:00,42,-292.94,0.0,0.0,-292.94,292.94,2026-06-24 06:00:13,56,72,2026-06-24 06:03:00
76030,2026-06-24,2026-06-24 00:45:00.613754+00:00,2026-06-24 14:30:00,58,-264.40,0.0,0.0,-264.40,264.40,2026-06-24 06:00:13,56,72,2026-06-24 06:03:00


In [28]:

def is_missing(value):
    return value is None or pd.isna(value)

def build_training_rows(df: pd.DataFrame):
    """
    Build training rows from the structured schedule DataFrame.
    """
    skipped_rows = []
    training_rows = []

    df = df.copy()
    df["time_block"] = df["time_block"].astype(int)
    df["time_retrieved_ist"] = pd.to_datetime(df["time_retrieved"]).dt.tz_convert("Asia/Kolkata")

    latest_retrieval = (
        df.groupby("schedule_date_ist")["time_retrieved_ist"].max().reset_index()
    )
    df_merged = df.merge(latest_retrieval, on=["schedule_date_ist", "time_retrieved_ist"], how="inner")

    grouped = df_merged.groupby(["schedule_date_ist", "time_retrieved_ist"])

    for (schedule_date_ist, time_retrieved_ist), group in grouped:

        schedule_date_only = pd.to_datetime(schedule_date_ist).date()

        print(f"schedule_date_ist: {schedule_date_ist}, time_retrieved_ist: {time_retrieved_ist}, group size: {len(group)}")

        for tb in range(1, 97):  # Time blocks from 1 to 96
            correct_timeblock_row = group[group["time_block"] == tb]
            if correct_timeblock_row.empty:
                skipped_rows.append({
                    "schedule_date_ist": schedule_date_ist,
                    "time_retrieved_ist": group["time_retrieved_ist"].iloc[0],
                    "prediction_tb": tb,
                    "reason": f"missing time block {tb}"
                })
            if len(correct_timeblock_row) != 1:
                skipped_rows.append({
                    "schedule_date_ist": schedule_date_ist,
                    "time_retrieved_ist": group["time_retrieved_ist"].iloc[0],
                    "prediction_tb": tb,
                    "reason": f"multiple rows for time block {tb}"
                })
                continue
            correct_timeblock_row = correct_timeblock_row.iloc[0]
            total_net_schd_amount = correct_timeblock_row.get("total_net_schd_amount")
            remc_station_sch_amount = correct_timeblock_row.get("remc_station_sch_amount")
            emergency_as = correct_timeblock_row.get("EMERGENCY_AS")
            shortfall_as = correct_timeblock_row.get("SHORTFALL_AS")
            training_rows.append({
                "prediction_for_time_block": tb,
                "prediction_for_period_end_ist": timeblock_to_period_end(schedule_date_only, tb),
                "schedule_date_ist": schedule_date_ist,
                "time_retrieved_ist": correct_timeblock_row["time_retrieved_ist"],
                "total_net_schd_amount": total_net_schd_amount,
                "remc_station_sch_amount": remc_station_sch_amount,
                "emergency_as": emergency_as,
                "shortfall_as": shortfall_as

            })
    #         vals = [
    #                 (target_tb_8, row_8.get("total_net_schd_amount"), row_8.get("remc_station_sch_amount"), row_8.get("emergency_as"), row_8.get("shortfall_as")),
    #                 (target_tb_9, row_9.get("total_net_schd_amount"), row_9.get("remc_station_sch_amount"), row_9.get("emergency_as"), row_9.get("shortfall_as"))
    #             ]

    #     for tb, total_net_schd_amount, remc_station_sch_amount, emergency_as, shortfall_as in vals:
    #         if tb is None or pd.isna(total_net_schd_amount) or pd.isna(remc_station_sch_amount) or pd.isna(emergency_as) or pd.isna(shortfall_as):
    #             skipped_rows.append({
    #                 "schedule_date_ist": schedule_date_ist,
    #                 "time_retrieved": time_retrieved_ist,
    #                 "prediction_tb": tb,
    #                 "reason": f"missing values for target_tb {tb}"
    #             })
    #             continue
            
    #         training_rows.append({
    #             "schedule_date_ist": schedule_date_ist,
    #             "prediction_for_time_block": tb,
    #             "prediction_for_period_end_ist": timeblock_to_period_end(schedule_date_only, tb),
    #             "time_retrieved": time_retrieved_ist,
    #             "time_retrieved_tb": retrieval_tb,
    #             "total_net_schd_amount": total_net_schd_amount,
    #             "remc_station_sch_amount": remc_station_sch_amount,
    #             "emergency_as": emergency_as,
    #             "shortfall_as": shortfall_as
    #         })
       

    return pd.DataFrame(training_rows), pd.DataFrame(skipped_rows)
    

In [29]:
training_df, skipped_df = build_training_rows(df_all_jsonb_rows)


schedule_date_ist: 2026-06-03, time_retrieved_ist: 2026-06-26 04:35:58.450269+05:30, group size: 96
schedule_date_ist: 2026-06-04, time_retrieved_ist: 2026-06-26 04:36:14.855063+05:30, group size: 96
schedule_date_ist: 2026-06-05, time_retrieved_ist: 2026-06-26 04:36:30.370298+05:30, group size: 96
schedule_date_ist: 2026-06-06, time_retrieved_ist: 2026-06-26 04:38:24.547184+05:30, group size: 96
schedule_date_ist: 2026-06-07, time_retrieved_ist: 2026-06-26 04:38:36.400809+05:30, group size: 96
schedule_date_ist: 2026-06-08, time_retrieved_ist: 2026-06-26 04:38:50.411422+05:30, group size: 96
schedule_date_ist: 2026-06-09, time_retrieved_ist: 2026-06-26 04:39:05.399749+05:30, group size: 96
schedule_date_ist: 2026-06-10, time_retrieved_ist: 2026-06-26 04:34:12.539239+05:30, group size: 96
schedule_date_ist: 2026-06-11, time_retrieved_ist: 2026-06-26 04:34:37.827571+05:30, group size: 96
schedule_date_ist: 2026-06-12, time_retrieved_ist: 2026-06-26 04:39:18.877288+05:30, group size: 96


In [30]:
display(training_df)
print("Training rows count:", len(training_df))
print("Skipped rows count:", len(skipped_df))


,prediction_for_time_block,prediction_for_period_end_ist,schedule_date_ist,time_retrieved_ist,total_net_schd_amount,remc_station_sch_amount,emergency_as,shortfall_as
0,1,2026-06-03 00:15:00,2026-06-03,2026-06-26 04:35:58.450269+05:30,0.0,0.0,0.0,0.0
1,2,2026-06-03 00:30:00,2026-06-03,2026-06-26 04:35:58.450269+05:30,0.0,0.0,0.0,0.0
2,3,2026-06-03 00:45:00,2026-06-03,2026-06-26 04:35:58.450269+05:30,0.0,0.0,0.0,0.0
3,4,2026-06-03 01:00:00,2026-06-03,2026-06-26 04:35:58.450269+05:30,0.0,0.0,0.0,0.0
4,5,2026-06-03 01:15:00,2026-06-03,2026-06-26 04:35:58.450269+05:30,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...
2107,92,2026-06-24 23:00:00,2026-06-24,2026-06-24 06:15:00.613754+05:30,0.0,0.0,0.0,0.0
2108,93,2026-06-24 23:15:00,2026-06-24,2026-06-24 06:15:00.613754+05:30,0.0,0.0,0.0,0.0
2109,94,2026-06-24 23:30:00,2026-06-24,2026-06-24 06:15:00.613754+05:30,0.0,0.0,0.0,0.0
2110,95,2026-06-24 23:45:00,2026-06-24,2026-06-24 06:15:00.613754+05:30,0.0,0.0,0.0,0.0


Training rows count: 2112
Skipped rows count: 0


In [ ]:
print(skipped_df['reason'].value_counts())
display(skipped_df.head(10))

reason
target_tb missing 55    18
target_tb missing 62    18
target_tb missing 60    18
target_tb missing 59    18
target_tb missing 58    18
                        ..
target_tb missing 13    14
target_tb missing 83     8
target_tb missing 9      8
target_tb missing 96     8
target_tb missing 82     8
Name: count, Length: 88, dtype: int64


,schedule_date_ist,time_retrieved,reason,resason
0,2026-06-15,2026-06-15 05:21:19.677954+00:00,target_tb missing 29,NaN
1,2026-06-15,2026-06-15 05:21:19.677954+00:00,target_tb missing 30,NaN
2,2026-06-15,2026-06-15 05:30:00.592747+00:00,target_tb missing 30,NaN
3,2026-06-15,2026-06-15 05:30:00.592747+00:00,target_tb missing 31,NaN
4,2026-06-15,2026-06-15 05:45:00.550275+00:00,target_tb missing 31,NaN
5,2026-06-15,2026-06-15 05:45:00.550275+00:00,target_tb missing 32,NaN
6,2026-06-15,2026-06-15 06:00:00.566680+00:00,target_tb missing 32,NaN
7,2026-06-15,2026-06-15 06:00:00.566680+00:00,target_tb missing 33,NaN
8,2026-06-15,2026-06-15 06:15:00.679685+00:00,target_tb missing 33,NaN
9,2026-06-15,2026-06-15 06:15:00.679685+00:00,target_tb missing 34,NaN


In [ ]:
# Save as CSV
datetime_str = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

schedule_csv_path = f"data/schedule_generation_training_data_{datetime_str}.csv"

training_df.to_csv(schedule_csv_path, index=False)

In [41]:
schedule_csv_path = "data/schedule_generation_training_data_20260625_221029.csv"
schedule_df = pd.read_csv(schedule_csv_path)

In [42]:
schedule_df

,prediction_for_time_block,prediction_for_period_end_ist,schedule_date_ist,time_retrieved_ist,total_net_schd_amount,remc_station_sch_amount,emergency_as,shortfall_as
0,1,2026-06-03 00:15:00,2026-06-03,2026-06-26 04:35:58.450269+05:30,0.0,0.0,0.0,0.0
1,2,2026-06-03 00:30:00,2026-06-03,2026-06-26 04:35:58.450269+05:30,0.0,0.0,0.0,0.0
2,3,2026-06-03 00:45:00,2026-06-03,2026-06-26 04:35:58.450269+05:30,0.0,0.0,0.0,0.0
3,4,2026-06-03 01:00:00,2026-06-03,2026-06-26 04:35:58.450269+05:30,0.0,0.0,0.0,0.0
4,5,2026-06-03 01:15:00,2026-06-03,2026-06-26 04:35:58.450269+05:30,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...
2107,92,2026-06-24 23:00:00,2026-06-24,2026-06-24 06:15:00.613754+05:30,0.0,0.0,0.0,0.0
2108,93,2026-06-24 23:15:00,2026-06-24,2026-06-24 06:15:00.613754+05:30,0.0,0.0,0.0,0.0
2109,94,2026-06-24 23:30:00,2026-06-24,2026-06-24 06:15:00.613754+05:30,0.0,0.0,0.0,0.0
2110,95,2026-06-24 23:45:00,2026-06-24,2026-06-24 06:15:00.613754+05:30,0.0,0.0,0.0,0.0


In [44]:


meter_csv_path = "data/meter_actual_generation_records_2026-06-28.csv"
meter_df = pd.read_csv(meter_csv_path)
meter_df_total = meter_df[meter_df['source'] == 'Total']
meter_df_total

,time_period_end,time_block,generation,source
59136,2026-03-30 00:15:00,1,-0.77,Total
59137,2026-03-30 00:30:00,2,-0.75,Total
59138,2026-03-30 00:45:00,3,-0.77,Total
59139,2026-03-30 01:00:00,4,-0.76,Total
59140,2026-03-30 01:15:00,5,-0.76,Total
...,...,...,...,...
66523,2026-06-14 23:00:00,92,-0.69,Total
66524,2026-06-14 23:15:00,93,-0.72,Total
66525,2026-06-14 23:30:00,94,-0.71,Total
66526,2026-06-14 23:45:00,95,-0.72,Total


In [ ]:
# combine schedule and meter csv

merged = pd.merge(schedule_df, meter_df_total, how='inner', left_on=['prediction_for_period_end_ist'], right_on=['time_period_end'])

In [54]:

merged_column_format = merged.copy()
merged_column_format.rename(columns={'generation': 'meter_actual_generation', 'source': 'meter_source', 'time_period_end': 'meter_time_period_end', 'time_block': 'meter_time_block'}, inplace=True)

In [55]:
merged_column_format

,prediction_for_time_block,prediction_for_period_end_ist,schedule_date_ist,time_retrieved_ist,total_net_schd_amount,remc_station_sch_amount,emergency_as,shortfall_as,meter_time_period_end,meter_time_block,meter_actual_generation,meter_source
0,1,2026-06-03 00:15:00,2026-06-03,2026-06-26 04:35:58.450269+05:30,0.0,0.0,0.0,0.0,2026-06-03 00:15:00,1,-0.76,Total
1,2,2026-06-03 00:30:00,2026-06-03,2026-06-26 04:35:58.450269+05:30,0.0,0.0,0.0,0.0,2026-06-03 00:30:00,2,-0.74,Total
2,3,2026-06-03 00:45:00,2026-06-03,2026-06-26 04:35:58.450269+05:30,0.0,0.0,0.0,0.0,2026-06-03 00:45:00,3,-0.75,Total
3,4,2026-06-03 01:00:00,2026-06-03,2026-06-26 04:35:58.450269+05:30,0.0,0.0,0.0,0.0,2026-06-03 01:00:00,4,-0.75,Total
4,5,2026-06-03 01:15:00,2026-06-03,2026-06-26 04:35:58.450269+05:30,0.0,0.0,0.0,0.0,2026-06-03 01:15:00,5,-0.75,Total
...,...,...,...,...,...,...,...,...,...,...,...,...
1147,92,2026-06-14 23:00:00,2026-06-14,2026-06-26 04:39:45.790730+05:30,0.0,0.0,0.0,0.0,2026-06-14 23:00:00,92,-0.69,Total
1148,93,2026-06-14 23:15:00,2026-06-14,2026-06-26 04:39:45.790730+05:30,0.0,0.0,0.0,0.0,2026-06-14 23:15:00,93,-0.72,Total
1149,94,2026-06-14 23:30:00,2026-06-14,2026-06-26 04:39:45.790730+05:30,0.0,0.0,0.0,0.0,2026-06-14 23:30:00,94,-0.71,Total
1150,95,2026-06-14 23:45:00,2026-06-14,2026-06-26 04:39:45.790730+05:30,0.0,0.0,0.0,0.0,2026-06-14 23:45:00,95,-0.72,Total


In [ ]:
merged_csv_path = f"data/merged_meter_total_vs_schedule.csv"


merged_column_format.to_csv(merged_csv_path, index=False)

: 